# 02 — Resume Category Classifier (Supervised)

Build and evaluate a multi-class text classifier that predicts the job domain from a resume.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Load Preprocessed Resumes

In [ ]:
from src.config import RESUME_PROC_CSV
from src.data.load_data import load_resumes
from src.data.preprocess import preprocess_resumes

try:
    df = pd.read_csv(RESUME_PROC_CSV)
    print("Loaded from cache:", df.shape)
except FileNotFoundError:
    raw_df = load_resumes()
    df     = preprocess_resumes(raw_df)
    print("Processed fresh:", df.shape)

print(df.head(2))

## 2. Train the Classifier

We compare **Logistic Regression**, **Linear SVM**, and **Random Forest** via 5-fold cross-validation and pick the best.

In [ ]:
from src.models.classifier import train_classifier

pipeline, results = train_classifier(df, text_col="clean_resume", label_col="Category")
print("Best model:", results["best_model"])

## 3. Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from src.config import RANDOM_STATE, TEST_SIZE

X = df["clean_resume"].values
y = df["Category"].values
_, X_test, _, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
y_pred = pipeline.predict(X_test)

fig, ax = plt.subplots(figsize=(14,10))
cm = confusion_matrix(y_test, y_pred, labels=sorted(set(y)))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=sorted(set(y)), yticklabels=sorted(set(y)), cmap="Blues", ax=ax)
ax.set_title("Confusion Matrix — Resume Classifier")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("../reports/figures/confusion_matrix_classifier.png", dpi=150)
plt.show()

## 4. Cross-Validation Summary

In [ ]:
cv_results = {k: v for k, v in results.items() if isinstance(v, dict) and "cv_mean" in v}
cv_df = pd.DataFrame(cv_results).T
print(cv_df)

## 5. Test Prediction on a Sample Resume

In [ ]:
sample_resume = """
Experienced Data Scientist with 4 years of experience in Python, machine learning,
deep learning, TensorFlow, PyTorch, SQL, and data visualization using Tableau.
Strong background in NLP and computer vision projects.
"""
from src.data.preprocess import clean_text
from src.models.classifier import predict_category

clean = clean_text(sample_resume)
predicted = predict_category(clean, pipeline)
print(f"Predicted Category: {predicted}")

## 6. Category Probability Bar Chart

In [ ]:
if hasattr(pipeline, "predict_proba"):
    proba  = pipeline.predict_proba([clean])[0]
    labels = pipeline.classes_
    top5   = sorted(zip(labels, proba), key=lambda x: -x[1])[:5]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh([l for l,p in top5], [p*100 for l,p in top5], color="#6366f1")
    ax.set_xlabel("Probability (%)")
    ax.set_title("Top-5 Category Probabilities")
    plt.tight_layout()
    plt.savefig("../reports/figures/category_probabilities.png", dpi=150)
    plt.show()

## ✅ Classifier complete. Next → 03_recommender.ipynb